# Praktikum 3 · Clustering: Struktur yang Bisa Dipertanggungjawabkan

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/__REPO_SLUG__/blob/main/notebooks/praktikum03_clustering.ipynb)

**EF235161 Topik Dalam Data Mining (P) · S-2 Teknik Informatika, ITS**
Clustering & Unsupervised Data Mining · pengerjaan asinkron, ±3 jam

---

## Isi identitas Anda

| | |
|---|---|
| **Nama** | _(isi)_ |
| **NRP** | _(isi)_ |
| **Username GitHub** | _(isi)_ |

---

## Pesan utama

> **Plot yang rapi bukan bukti. Cluster baru layak diklaim bila mengalahkan lantai tanpa struktur, stabil, dan dipilih tanpa mengintip label.**

Praktikum 2 mengajarkan bahwa klaim "lebih baik" hanya sekuat baseline-nya. Di clustering tidak ada label untuk dicocokkan, jadi pertanyaannya bergeser: *apakah struktur yang Anda temukan benar ada, atau hanya bentuk yang dipaksakan algoritma?*

## Cara menyimpan pekerjaan Anda

Bekerja di Colab, **sekali di awal**:

1. Klik **Copy to Drive** di bar atas. Sebelum ini, pilihan simpan ke GitHub belum muncul.
2. `File → Open notebook → tab GitHub`, centang **Include private repositories**, lalu *Authorize*.

Setiap kali ingin menyimpan: `File → Save a copy in GitHub` → repositori praktikum Anda → branch `main` → **pastikan path-nya** `notebooks/praktikum03_clustering.ipynb`. Itulah commit Anda.

Bila tetap gagal: `File → Download → Download .ipynb`, lalu unggah lewat `Add file → Upload files` di repositori Anda (folder `notebooks/`, nama sama).

## Aturan

- Jalankan sel **berurutan dari atas**. Jangan melompat.
- Setelah setiap blok, isi satu sel **Interpretasi**. Menjalankan kode tanpa interpretasi berarti belum mengerjakan praktikum.
- **Label referensi dikunci sampai Blok 8.** Di penelitian nyata label itu tidak ada; memakainya untuk memilih `k` atau `eps` sama dengan membuka test set lebih awal.
- **Anggaran pencarian sama untuk semua algoritma** (`ANGGARAN_PARAM`), dan **preprocessing sama** untuk semua (`buat_pipeline`).
- Tugas kode Anda ada di **Blok 4**: menambah satu algoritma dari keluarga berbeda.
- Runtime penuh sekitar 1 sampai 2 menit di Colab.

## Blok 0 · Setup & reproducibility

In [ ]:
import sys, os, time, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import scipy, sklearn
from scipy import stats

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
os.makedirs("outputs", exist_ok=True)
T_MULAI = time.time()

print("Python      :", sys.version.split()[0])
print("NumPy       :", np.__version__)
print("pandas      :", pd.__version__)
print("SciPy       :", scipy.__version__)
print("scikit-learn:", sklearn.__version__)
print("RANDOM_STATE:", RANDOM_STATE)

Versi library dan seed dicatat sebelum analisis apa pun. K-Means, t-SNE, dan pencarian parameter bergantung pada seed.

**Interpretasi** (1 sampai 2 kalimat, sebut angkanya): _(tulis di sini)_

## Blok 1 · Enam dataset, enam karakter

Tidak ada algoritma clustering yang terbaik untuk semua data. Karena itu kita memakai dataset dengan karakter berbeda:

| Kode | Karakter | Pertanyaan yang diuji |
|---|---|---|
| **A** | Globular: tiga gumpalan bulat | Kasus ideal. Apakah semua algoritma berhasil? |
| **B** | Non-globular: dua bulan sabit | Bagaimana algoritma berbasis centroid menangani bentuk melengkung? |
| **C** | Tiga gumpalan + 10% titik noise | Apa yang terjadi pada titik yang bukan milik cluster mana pun? |
| **D** | Kepadatan dan skala fitur berbeda | Apa peran standardisasi? Bagaimana satu ambang kepadatan menghadapi kepadatan berbeda? |
| **E** | Wine (data nyata, 13 fitur) | Bagaimana algoritma bertahan di dimensi lebih tinggi? |
| **K** | Kontrol: titik acak seragam, **tanpa struktur** | Berapa skor yang didapat algoritma ketika sebenarnya tidak ada cluster? |

Label referensi disimpan di `Y_REFERENSI` dan **dikunci sampai Blok 8**. Dataset A sampai D dan K sintetis; lihat `data/DATA_CARD.md`.

In [ ]:
from sklearn.datasets import make_blobs, make_moons, load_wine

DATASET = {}          # nama -> matriks fitur (tanpa label)
Y_REFERENSI = {}      # nama -> label referensi, DIKUNCI sampai Blok 8

_X, _y = make_blobs(n_samples=600, centers=3, cluster_std=1.0, random_state=RANDOM_STATE)
DATASET["A globular"], Y_REFERENSI["A globular"] = _X, _y

_X, _y = make_moons(n_samples=600, noise=0.08, random_state=RANDOM_STATE)
DATASET["B bulan sabit"], Y_REFERENSI["B bulan sabit"] = _X, _y

_Xc, _yc = make_blobs(n_samples=540, centers=3, cluster_std=0.7, random_state=7)
_Xn = np.random.default_rng(RANDOM_STATE).uniform(_Xc.min(0) - 2, _Xc.max(0) + 2, size=(60, 2))
DATASET["C noise"] = np.vstack([_Xc, _Xn])
Y_REFERENSI["C noise"] = np.r_[_yc, -np.ones(60, dtype=int)]      # -1 = noise

_X, _y = make_blobs(n_samples=[300, 200, 100], centers=[[0, 0], [6, 0], [3, 7]],
                    cluster_std=[0.4, 1.2, 2.2], random_state=RANDOM_STATE)
_X[:, 1] *= 50                                                     # fitur 2 dalam satuan lain
DATASET["D kepadatan & skala"], Y_REFERENSI["D kepadatan & skala"] = _X, _y

_w = load_wine()
DATASET["E wine (nyata)"], Y_REFERENSI["E wine (nyata)"] = _w.data, _w.target
NAMA_FITUR_WINE = list(_w.feature_names)

DATASET["K kontrol acak"] = np.random.default_rng(RANDOM_STATE).uniform(0, 1, size=(600, 2))
Y_REFERENSI["K kontrol acak"] = np.zeros(600, dtype=int)             # satu kelompok: tidak ada struktur
del _X, _y, _Xc, _yc, _Xn, _w

for nama, X in DATASET.items():
    print(f"{nama:22s} ukuran={X.shape}  rentang fitur (min..maks per kolom, 2 pertama): "
          + ", ".join(f"[{X[:, j].min():.1f}..{X[:, j].max():.1f}]" for j in range(2)))

fig, axes = plt.subplots(1, 5, figsize=(17, 3.2))
for ax, nama in zip(axes, [n for n in DATASET if DATASET[n].shape[1] == 2]):
    X = DATASET[nama]
    ax.scatter(X[:, 0], X[:, 1], s=5, color="0.35")
    ax.set_title(nama, fontsize=10)
plt.suptitle("Data mentah tanpa warna label: inilah yang dilihat peneliti", y=1.03)
plt.tight_layout(); plt.savefig("outputs/dataset_mentah.png", dpi=130, bbox_inches="tight"); plt.show()

Lihat rentang fitur Dataset D dan plot mentahnya. Apa yang akan terjadi pada jarak Euclid tanpa standardisasi?

**Interpretasi** (1 sampai 2 kalimat, sebut angkanya): _(tulis di sini)_

## Blok 2 · Protokol yang adil (dibaca dulu, sebelum menjalankan)

Empat aturan untuk **semua** algoritma, tanpa pengecualian:

1. **Label dikunci.** Konfigurasi dipilih hanya dengan kriteria internal (tanpa label). Label dibuka satu kali di Blok 8 untuk menilai pilihan itu.
2. **Preprocessing sama.** Setiap algoritma dijalankan lewat `buat_pipeline`: `StandardScaler` lalu algoritmanya.
3. **Anggaran pencarian setara.** Setiap algoritma mencoba `ANGGARAN_PARAM` konfigurasi, diambil dengan `ParameterSampler` yang sama.
4. **Aturan pilih yang ditetapkan di muka.** Pilih silhouette tertinggi di antara konfigurasi yang valid: minimal 2 cluster dan noise paling banyak `BATAS_NOISE`. Aturan ini tidak boleh diubah setelah melihat hasil.

Silhouette untuk algoritma yang menghasilkan noise (label `-1`) dihitung hanya pada titik non-noise; karena itu fraksi noise selalu dilaporkan di sampingnya.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import ParameterSampler
from sklearn.metrics import silhouette_score, davies_bouldin_score

ANGGARAN_PARAM = 8          # sama untuk SEMUA algoritma
BATAS_NOISE = 0.5           # konfigurasi dengan noise > 50% dianggap tidak valid
KRITERIA = "silhouette"     # kriteria internal untuk memilih; alasannya dibahas di laporan


def buat_pipeline(estimator):
    """Satu-satunya jalur preprocessing: standardisasi lalu algoritma."""
    return Pipeline([("scaler", StandardScaler()), ("clu", estimator)])


def label_dari(pipa, X):
    """Fit lalu kembalikan label. GaussianMixture tidak punya fit_predict pada Pipeline lama, jadi ditangani."""
    try:
        return np.asarray(pipa.fit_predict(X))
    except AttributeError:
        return np.asarray(pipa.fit(X).predict(X))


def skor_internal(Xs, label):
    """Metrik tanpa label, dihitung di ruang yang sudah distandardisasi."""
    inti = label != -1
    k = len(set(label[inti]))
    noise = 1 - inti.mean()
    if k < 2 or inti.sum() < 10:
        return {"k": k, "noise": noise, "silhouette": np.nan, "davies_bouldin": np.nan}
    return {"k": k, "noise": noise,
            "silhouette": silhouette_score(Xs[inti], label[inti]),
            "davies_bouldin": davies_bouldin_score(Xs[inti], label[inti])}


def ruang_standar(X):
    return buat_pipeline(None).named_steps["scaler"].fit_transform(X)

print("ANGGARAN_PARAM:", ANGGARAN_PARAM, "| BATAS_NOISE:", BATAS_NOISE, "| KRITERIA:", KRITERIA)

Mengapa memilih `k` atau `eps` dengan melihat label sama bermasalahnya dengan tuning di test set? Tulis satu kalimat.

**Interpretasi** (1 sampai 2 kalimat, sebut angkanya): _(tulis di sini)_

## Blok 3 · Lantai: skor yang didapat tanpa struktur apa pun

Sebelum menilai algoritma, ukur dulu skor yang didapat pada data **tanpa struktur**: titik acak seragam dengan ukuran dan dimensi yang sama. K-Means dijalankan dengan k = 2 sampai 9 dan silhouette terbaiknya dicatat, diulang 5 seed. Skor sebuah dataset baru bermakna bila jelas di atas lantainya sendiri.

In [ ]:
from sklearn.cluster import KMeans

def lantai_silhouette(X, ulangan=5, k_maks=9):
    """Silhouette K-Means terbaik pada data seragam acak di kotak batas X (sudah distandardisasi)."""
    Xs = ruang_standar(X)
    hasil = []
    for r in range(ulangan):
        U = np.random.default_rng(RANDOM_STATE + r).uniform(Xs.min(0), Xs.max(0), size=Xs.shape)
        hasil.append(max(silhouette_score(U, KMeans(k, n_init=10, random_state=RANDOM_STATE).fit_predict(U))
                         for k in range(2, k_maks + 1)))
    return np.mean(hasil), np.std(hasil)

LANTAI = {}
for nama, X in DATASET.items():
    LANTAI[nama] = lantai_silhouette(X)
    print(f"{nama:22s} lantai silhouette = {LANTAI[nama][0]:.3f} ± {LANTAI[nama][1]:.3f}")

Berapa silhouette yang didapat K-Means pada data yang **tidak punya** cluster? Mengapa lantai Dataset E berbeda jauh dari yang lain?

**Interpretasi** (1 sampai 2 kalimat, sebut angkanya): _(tulis di sini)_

## Blok 4 · Ruang pencarian dan **tugas kode Anda**

Dua algoritma wajib:

- **K-Means** (berbasis centroid): mencoba `n_clusters` 2 sampai 9.
- **DBSCAN** (berbasis kepadatan): mencoba `min_samples` dan `eps`. Karena skala jarak berbeda antar dataset (bandingkan 2 dimensi dengan 13 dimensi), `eps` tidak diberikan sebagai angka tetap, melainkan sebagai **kuantil k-distance** data itu sendiri, antara 85% dan 99,5%. Inilah heuristik yang diusulkan penulis DBSCAN: lutut *k-distance plot* adalah kandidat `eps`.

### Tugas: tambahkan satu algoritma dari keluarga berbeda

Isi `ALGORITMA_SAYA` dengan **satu atau lebih** algoritma lain (Agglomerative, Gaussian Mixture, HDBSCAN, OPTICS, Spectral, atau lainnya yang bisa Anda pertanggungjawabkan). Setiap entri berisi `(fungsi_pembuat, ruang_parameter)`. **Jangan** menambahkan scaler sendiri: `buat_pipeline` sudah melakukannya, dan preprocessing harus sama untuk semua. Anggaran pencarian otomatis `ANGGARAN_PARAM`.

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors

from sklearn.base import BaseEstimator, ClusterMixin

KUANTIL_EPS = (0.85, 0.995)     # rentang eps sebagai kuantil k-distance


class DBSCANkDist(ClusterMixin, BaseEstimator):
    """DBSCAN dengan eps = kuantil jarak ke tetangga ke-min_samples pada data yang sedang di-fit."""
    def __init__(self, kuantil=0.95, min_samples=10):
        self.kuantil, self.min_samples = kuantil, min_samples

    def fit(self, X, y=None):
        jarak = NearestNeighbors(n_neighbors=self.min_samples).fit(X).kneighbors(X)[0][:, -1]
        self.eps_ = float(np.quantile(jarak, self.kuantil))
        self.labels_ = DBSCAN(eps=self.eps_, min_samples=self.min_samples).fit(X).labels_
        return self


# k-distance plot: jarak ke tetangga ke-10, diurutkan. Lutut kurva = kandidat eps.
fig, axes = plt.subplots(1, len(DATASET), figsize=(18, 2.8))
for ax, (nama, X) in zip(axes, DATASET.items()):
    Xs = ruang_standar(X)
    jarak = np.sort(NearestNeighbors(n_neighbors=10).fit(Xs).kneighbors(Xs)[0][:, -1])
    ax.plot(jarak); ax.set_title(nama, fontsize=9)
    ax.axhspan(*np.quantile(jarak, KUANTIL_EPS), color="orange", alpha=0.25)
axes[0].set_ylabel("jarak ke tetangga ke-10")
plt.suptitle("k-distance plot (pita oranye = rentang eps yang dicari, kuantil 85% sampai 99,5%)", y=1.05)
plt.tight_layout(); plt.savefig("outputs/k_distance.png", dpi=130, bbox_inches="tight"); plt.show()

ALGORITMA = {
    "K-Means": (lambda **p: KMeans(n_init=10, random_state=RANDOM_STATE, **p),
                {"n_clusters": list(range(2, 10))}),
    "DBSCAN": (lambda **p: DBSCANkDist(**p),
               {"kuantil": stats.uniform(KUANTIL_EPS[0], KUANTIL_EPS[1] - KUANTIL_EPS[0]),
                "min_samples": [5, 10, 20]}),
}

In [ ]:
from sklearn.cluster import AgglomerativeClustering, HDBSCAN
from sklearn.mixture import GaussianMixture

# --- Isi di sini. Hapus tanda # pada contoh lalu sesuaikan, atau tulis algoritma Anda sendiri. ---
# ALGORITMA_SAYA = {
#     "Agglomerative (ward)": (
#         lambda **p: AgglomerativeClustering(linkage="ward", **p),
#         {"n_clusters": list(range(2, 10))},
#     ),
# }
ALGORITMA_SAYA = {}

Di dataset mana k-distance plot punya lutut yang jelas, dan di mana kurvanya naik landai tanpa lutut? Apa artinya bagi DBSCAN? Mengapa Anda memilih algoritma tambahan itu?

**Interpretasi** (1 sampai 2 kalimat, sebut angkanya): _(tulis di sini)_

## Blok 5 · Pencarian dengan anggaran setara, dipilih tanpa label

Setiap algoritma mencoba `ANGGARAN_PARAM` konfigurasi di setiap dataset. Konfigurasi terpilih mengikuti aturan Blok 2. Tabel di bawah juga menampilkan lantai dari Blok 3. Anda tidak perlu mengubah sel ini.

In [ ]:
def cari_konfigurasi(pembuat, ruang, X):
    """Coba ANGGARAN_PARAM konfigurasi, nilai dengan metrik internal. Tidak menyentuh label."""
    Xs = ruang_standar(X)
    baris = []
    for param in ParameterSampler(ruang, n_iter=ANGGARAN_PARAM, random_state=RANDOM_STATE):
        label = label_dari(buat_pipeline(pembuat(**param)), X)
        baris.append({"param": param, "label": label, **skor_internal(Xs, label)})
    return baris


def pilih(baris):
    valid = [b for b in baris if b["k"] >= 2 and b["noise"] <= BATAS_NOISE and not np.isnan(b[KRITERIA])]
    return max(valid, key=lambda b: b[KRITERIA]) if valid else None


SEMUA_ALGORITMA = {**ALGORITMA, **ALGORITMA_SAYA}
HASIL_CARI, PILIHAN = {}, {}
for nama, X in DATASET.items():
    for alg, (pembuat, ruang) in SEMUA_ALGORITMA.items():
        HASIL_CARI[(nama, alg)] = cari_konfigurasi(pembuat, ruang, X)
        PILIHAN[(nama, alg)] = pilih(HASIL_CARI[(nama, alg)])


def fmt_param(p):
    return ", ".join(f"{k}={v:.3f}" if isinstance(v, float) else f"{k}={v}" for k, v in p.items())

TABEL_PILIHAN = pd.DataFrame([
    {"dataset": nama, "algoritma": alg,
     "param terpilih": fmt_param(b["param"]) if b else "tidak ada yang valid",
     "k": b["k"] if b else 0, "noise": round(b["noise"], 3) if b else np.nan,
     "silhouette": round(b["silhouette"], 3) if b else np.nan,
     "davies_bouldin": round(b["davies_bouldin"], 3) if b else np.nan,
     "lantai": round(LANTAI[nama][0], 3)}
    for (nama, alg), b in PILIHAN.items()])
print(TABEL_PILIHAN.to_string(index=False))
print(f"\nAlgoritma: {list(SEMUA_ALGORITMA)} | anggaran {ANGGARAN_PARAM} konfigurasi per algoritma per dataset")

Bandingkan silhouette terpilih dengan lantainya. Dataset mana yang skornya tinggi tetapi tidak jauh dari lantai? Di dataset mana K-Means dan DBSCAN memilih jumlah cluster berbeda?

**Interpretasi** (1 sampai 2 kalimat, sebut angkanya): _(tulis di sini)_

## Blok 6 · Sensitivitas dan stabilitas

Dua pertanyaan tanpa label:

1. **Sensitivitas:** apakah kesimpulan berubah bila parameter sedikit berbeda? Kurva silhouette K-Means terhadap `k` ditampilkan untuk semua dataset, bukan hanya titik terbaiknya.
2. **Stabilitas:** jalankan konfigurasi terpilih pada dua subsampel 80% yang berbeda, lalu bandingkan label pada titik yang sama dengan ARI. Diulang 10 kali. Ini tidak memakai label referensi; yang dibandingkan adalah dua hasil clustering.

In [ ]:
from sklearn.metrics import adjusted_rand_score

fig, ax = plt.subplots(figsize=(7, 4))
for nama in DATASET:
    b = sorted(HASIL_CARI[(nama, "K-Means")], key=lambda r: r["param"]["n_clusters"])
    ax.plot([r["param"]["n_clusters"] for r in b], [r["silhouette"] for r in b], marker="o", label=nama)
ax.set_xlabel("n_clusters"); ax.set_ylabel("silhouette"); ax.set_title("Sensitivitas K-Means terhadap k")
ax.legend(fontsize=8); plt.tight_layout(); plt.savefig("outputs/sensitivitas_kmeans.png", dpi=130); plt.show()


def stabilitas(pembuat, param, X, ulangan=10, porsi=0.8):
    """ARI antar dua clustering pada subsampel berbeda, dihitung pada titik yang muncul di keduanya."""
    rng = np.random.default_rng(RANDOM_STATE)
    n, skor = len(X), []
    for _ in range(ulangan):
        a = rng.choice(n, int(porsi * n), replace=False)
        b = rng.choice(n, int(porsi * n), replace=False)
        la, lb = np.full(n, -2), np.full(n, -2)
        la[a] = label_dari(buat_pipeline(pembuat(**param)), X[a])
        lb[b] = label_dari(buat_pipeline(pembuat(**param)), X[b])
        sama = np.intersect1d(a, b)
        skor.append(adjusted_rand_score(la[sama], lb[sama]))
    return np.mean(skor), np.std(skor)

STABIL = {}
for (nama, alg), b in PILIHAN.items():
    if b is not None:
        STABIL[(nama, alg)] = stabilitas(SEMUA_ALGORITMA[alg][0], b["param"], DATASET[nama])
print(pd.DataFrame([{"dataset": n, "algoritma": a, "stabilitas (ARI antar subsampel)": f"{m:.3f} ± {s:.3f}"}
                    for (n, a), (m, s) in STABIL.items()]).to_string(index=False))

Apakah konfigurasi terpilih pada Dataset K (tanpa struktur) stabil? Apa artinya bagi klaim "cluster kami stabil, jadi bermakna"?

**Interpretasi** (1 sampai 2 kalimat, sebut angkanya): _(tulis di sini)_

## Blok 7 · Jebakan visual

Dua percobaan:

1. **PCA** Dataset E (13 dimensi) diproyeksikan ke 2D, diwarnai hasil K-Means terpilih. Berapa persen variansi yang tertangkap dua komponen itu?
2. **t-SNE pada data acak**: 600 titik acak normal 13 dimensi (tanpa struktur). Lalu K-Means dijalankan **pada hasil t-SNE**, kebiasaan yang sering muncul di paper. Bandingkan silhouette di ruang t-SNE dengan silhouette label yang sama di ruang aslinya.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

nama_e = "E wine (nyata)"
Xs_e = ruang_standar(DATASET[nama_e])
pca = PCA(n_components=2, random_state=RANDOM_STATE).fit(Xs_e)
Z_e = pca.transform(Xs_e)
lab_e = PILIHAN[(nama_e, "K-Means")]["label"]

ACAK = np.random.default_rng(RANDOM_STATE).normal(size=(600, 13))
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
axes[0].scatter(Z_e[:, 0], Z_e[:, 1], c=lab_e, s=10, cmap="tab10")
axes[0].set_title(f"PCA {nama_e}, variansi 2 komponen = {pca.explained_variance_ratio_.sum():.0%}", fontsize=9)
SIL_TSNE = {}
for ax, perp in zip(axes[1:], [5, 30, 50]):
    Z = TSNE(perplexity=perp, random_state=RANDOM_STATE).fit_transform(ACAK)
    lab = KMeans(6, n_init=10, random_state=RANDOM_STATE).fit_predict(Z)
    SIL_TSNE[perp] = (silhouette_score(Z, lab), silhouette_score(ACAK, lab))
    ax.scatter(Z[:, 0], Z[:, 1], c=lab, s=5, cmap="tab10")
    ax.set_title(f"t-SNE data ACAK, perplexity={perp}", fontsize=9)
plt.tight_layout(); plt.savefig("outputs/jebakan_visual.png", dpi=130); plt.show()

for perp, (s_emb, s_asli) in SIL_TSNE.items():
    print(f"perplexity={perp:2d}  silhouette di ruang t-SNE={s_emb:.3f}  |  label yang sama di ruang asli={s_asli:.3f}")

Plot t-SNE data acak terlihat punya enam kelompok berwarna. Apa yang dikatakan dua angka silhouette itu? Apa pelajarannya untuk gambar t-SNE/UMAP di paper?

**Interpretasi** (1 sampai 2 kalimat, sebut angkanya): _(tulis di sini)_

## Blok 8 · Buka label **satu kali**

Konfigurasi sudah dipilih di Blok 5 tanpa label. Sekarang label referensi dibuka untuk menilai pilihan itu dengan **ARI** dan **NMI**. Sel ini dan sel sesudahnya satu-satunya tempat `Y_REFERENSI` dibaca.

Kolom **ARI terbaik dalam anggaran** adalah ARI tertinggi di antara `ANGGARAN_PARAM` konfigurasi yang sudah dicoba. Angka itu **tidak bisa** didapat di penelitian nyata karena memilihnya butuh label. Selisihnya dengan ARI terpilih adalah *harga tidak punya label*.

In [ ]:
from sklearn.metrics import normalized_mutual_info_score

NAMA_KONTROL = "K kontrol acak"     # referensinya satu kelompok, jadi ARI/NMI tidak bermakna di sini
baris = []
for (nama, alg), b in PILIHAN.items():
    sil = round(b["silhouette"], 3) if b else np.nan
    if nama == NAMA_KONTROL:
        baris.append({"dataset": nama, "algoritma": alg, "silhouette": sil, "ARI terpilih": np.nan,
                      "NMI terpilih": np.nan, "ARI terbaik dalam anggaran": np.nan, "harga tanpa label": np.nan})
        continue
    y = Y_REFERENSI[nama]
    ari = adjusted_rand_score(y, b["label"]) if b else np.nan        # NaN: tidak ada konfigurasi valid
    nmi = normalized_mutual_info_score(y, b["label"]) if b else np.nan
    ari_maks = max(adjusted_rand_score(y, r["label"]) for r in HASIL_CARI[(nama, alg)]) if b else np.nan
    baris.append({"dataset": nama, "algoritma": alg, "silhouette": sil,
                  "ARI terpilih": round(ari, 3), "NMI terpilih": round(nmi, 3),
                  "ARI terbaik dalam anggaran": round(ari_maks, 3), "harga tanpa label": round(ari_maks - ari, 3)})
TABEL_LABEL = pd.DataFrame(baris)
print(TABEL_LABEL.to_string(index=False))
print("NaN pada baris selain Dataset K berarti algoritma itu tidak menemukan konfigurasi yang valid (lihat Blok 5).")

print("\nPeran standardisasi pada Dataset D, K-Means k=3:")
X_d, y_d = DATASET["D kepadatan & skala"], Y_REFERENSI["D kepadatan & skala"]
print("  tanpa standardisasi ARI =", round(adjusted_rand_score(y_d, KMeans(3, n_init=10, random_state=RANDOM_STATE).fit_predict(X_d)), 3))
print("  dengan buat_pipeline ARI =", round(adjusted_rand_score(y_d, label_dari(buat_pipeline(KMeans(3, n_init=10, random_state=RANDOM_STATE)), X_d)), 3))
print("\nDataset K tidak punya struktur, jadi tidak ada label yang bisa dicocokkan (NaN). Nilainya ada di Blok 3, 5, dan 6.")

Pada dataset mana silhouette dan ARI memberi peringkat algoritma yang berlawanan? Di mana harga tidak punya label paling besar, dan mengapa?

**Interpretasi** (1 sampai 2 kalimat, sebut angkanya): _(tulis di sini)_

## Blok 9 · Profil cluster pada data nyata

Clustering yang berguna harus bisa dijelaskan. Untuk Dataset E, hitung rata-rata tiap fitur (dalam satuan simpangan baku) per cluster K-Means terpilih, lalu beri nama sementara pada tiap cluster dari fitur yang paling membedakan. Setelah itu (label sudah dibuka), bandingkan dengan varietas anggur yang sebenarnya.

In [ ]:
profil = pd.DataFrame(Xs_e, columns=NAMA_FITUR_WINE).assign(cluster=lab_e).groupby("cluster").mean().round(2)
ukuran = pd.Series(lab_e).value_counts().sort_index()
print("Ukuran cluster:", dict(ukuran))
print("\nRata-rata fitur per cluster (z-score):")
print(profil.T.to_string())
print("\nTiga fitur paling membedakan per cluster (|z| terbesar):")
for c, r in profil.iterrows():
    top = r.abs().sort_values(ascending=False).index[:3]
    print(f"  cluster {c}: " + ", ".join(f"{f} ({r[f]:+.2f})" for f in top))
print("\nTabulasi silang cluster vs varietas (label referensi):")
print(pd.crosstab(pd.Series(lab_e, name="cluster"), pd.Series(Y_REFERENSI[nama_e], name="varietas")))

Beri nama sementara pada tiap cluster dari profilnya. Apakah cluster itu sesuai dengan varietas? Apakah profil saja cukup membuktikan cluster bermakna?

**Interpretasi** (1 sampai 2 kalimat, sebut angkanya): _(tulis di sini)_

## Blok 10 · Ringkasan lintas dataset

In [ ]:
algs = list(SEMUA_ALGORITMA)
fig, ax = plt.subplots(figsize=(11, 4))
lebar = 0.8 / len(algs)
berlabel = [n for n in DATASET if n != NAMA_KONTROL]
posisi = np.arange(len(berlabel))
for i, alg in enumerate(algs):
    nilai = [TABEL_LABEL[(TABEL_LABEL.dataset == n) & (TABEL_LABEL.algoritma == alg)]["ARI terpilih"].iloc[0] for n in berlabel]
    ax.bar(posisi + i * lebar, nilai, lebar, label=alg)
ax.set_xticks(posisi + lebar * (len(algs) - 1) / 2); ax.set_xticklabels(berlabel, rotation=15, fontsize=9)
ax.set_ylabel("ARI konfigurasi terpilih"); ax.set_title("Tidak ada algoritma yang terbaik untuk semua data")
ax.legend(); plt.tight_layout(); plt.savefig("outputs/ringkasan_ari.png", dpi=130); plt.show()

print("Pemenang per dataset (Dataset K tidak berlaku):")
for nama in berlabel:
    t = TABEL_LABEL[TABEL_LABEL.dataset == nama]
    menurut_sil = t.sort_values("silhouette", ascending=False).iloc[0]["algoritma"]
    menurut_ari = t.sort_values("ARI terpilih", ascending=False).iloc[0]["algoritma"]
    print(f"  {nama:22s} menurut silhouette: {menurut_sil:22s} menurut ARI: {menurut_ari}")

Apakah pemenang menurut silhouette sama dengan pemenang menurut ARI? Karakter data apa yang menentukan algoritma mana yang cocok?

**Interpretasi** (1 sampai 2 kalimat, sebut angkanya): _(tulis di sini)_

## Blok 11 · Log reproduksibilitas

In [ ]:
print("scikit-learn      :", sklearn.__version__)
print("RANDOM_STATE      :", RANDOM_STATE)
print("ANGGARAN_PARAM    :", ANGGARAN_PARAM)
print("BATAS_NOISE       :", BATAS_NOISE)
print("Kriteria pilih    :", KRITERIA, "(tanpa label)")
print("Stabilitas        : 10 pasang subsampel 80%")
print("Algoritma tambahan:", list(ALGORITMA_SAYA) or "BELUM DIISI")
print(f"Waktu total       : {(time.time()-T_MULAI)/60:.1f} menit")

## Opsional (nilai tambah) · Jalankan protokol pada data penelitian Anda

Tidak diperiksa otomatis, tetapi **dinilai sebagai nilai tambah** (lihat `docs/RUBRIK.md`). Isi `X_riset` dengan matriks fitur dari penelitian Anda: fitur tabular, embedding teks atau gambar, atau representasi lain. Fungsi di bawah menjalankan seluruh protokol **tanpa label**: lantai, pencarian beranggaran setara, dan stabilitas. Tulis hasil dan satu paragraf interpretasinya di Bagian 6 laporan.

Bila data Anda besar, ambil subsampel acak (misalnya 2.000 baris) dan sebutkan itu di laporan.

In [ ]:
def jalankan_protokol(X_riset, nama="riset saya"):
    X_riset = np.asarray(X_riset, dtype=float)
    lt = lantai_silhouette(X_riset)
    baris = []
    for alg, (pembuat, ruang) in SEMUA_ALGORITMA.items():
        b = pilih(cari_konfigurasi(pembuat, ruang, X_riset))
        st = stabilitas(pembuat, b["param"], X_riset) if b else (np.nan, np.nan)
        baris.append({"algoritma": alg, "param": fmt_param(b["param"]) if b else "tidak ada yang valid",
                      "k": b["k"] if b else 0, "noise": round(b["noise"], 3) if b else np.nan,
                      "silhouette": round(b["silhouette"], 3) if b else np.nan,
                      "lantai": f"{lt[0]:.3f} ± {lt[1]:.3f}", "stabilitas": f"{st[0]:.3f} ± {st[1]:.3f}"})
    print(f"Protokol pada {nama}, ukuran {X_riset.shape}")
    return pd.DataFrame(baris)

# Contoh: ganti baris di bawah dengan data Anda, lalu jalankan.
# X_riset = pd.read_csv("data_saya.csv").select_dtypes("number").dropna().to_numpy()
# jalankan_protokol(X_riset, "nama dataset saya")

## Selesai · langkah berikutnya

1. Pastikan semua sel sudah dijalankan (**Runtime → Run all**), lalu simpan ke GitHub.
2. Isi `laporan/LAPORAN.md`: empat temuan, Kartu Validitas Struktur, audit klaim visual, refleksi, dan penggunaan AI.
3. Push. Buka tab **Actions** dan pastikan pemeriksaan berwarna hijau.